# U-Net++ EfficientNet-B4 — Segmentación de Vértebras

Segmentación semántica multiclase de vértebras en radiografías de columna.  
**Clases:** 17 (T1–T12 + L1–L5) + background  
**Arquitectura:** U-Net++ con encoder EfficientNet-B4 (pretrained ImageNet)  
**Por qué es mejor que Mask R-CNN para este problema:**
- Segmentación densa pixel-a-pixel (no masks interpoladas 28×28)
- El encoder EfficientNet-B4 entiende características finas de textura
- U-Net++ tiene skip connections densas que preservan bordes de vértebras
- Entrenamiento más estable con 250 imágenes

> **Hardware objetivo:** Google Colab T4/A100 — FP16 habilitado

## 0 — Instalación de dependencias

In [ ]:
!pip install -q segmentation-models-pytorch albumentations timm

## 1 — Imports y Configuración

In [ ]:
import os
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
import random
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ===================== CONFIGURACIÓN =====================
DRIVE_ROOT       = Path("/content/drive/MyDrive")
DATASET_ROOT     = DRIVE_ROOT / "Scoliosis_Dataset"
DATASET_INDEX    = os.path.join(DATASET_ROOT, 'indice_dataset.csv')
CHECKPOINTS_DIR  = 'checkpoints_unetpp'

DEVICE       = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
TARGET_SIZE  = (512, 1024)   # (width, height) — mismo que antes
NUM_CLASSES  = 18            # 0=background + 17 vértebras
BASE_LR      = 3e-4          # Adam es más agresivo que SGD; 3e-4 es el punto de partida
WEIGHT_DECAY = 1e-5
BATCH_SIZE   = 4             # U-Net++ es más eficiente en memoria que MaskRCNN
NUM_EPOCHS   = 100
PATIENCE     = 15            # early stopping
SEED         = 42

CLASS_NAMES = {
    1: 'T1',  2: 'T2',  3: 'T3',  4: 'T4',   5: 'T5',
    6: 'T6',  7: 'T7',  8: 'T8',  9: 'T9',  10: 'T10',
    11: 'T11', 12: 'T12',
    13: 'L1', 14: 'L2', 15: 'L3', 16: 'L4', 17: 'L5',
}

os.makedirs(CHECKPOINTS_DIR, exist_ok=True)
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
print(f'Dispositivo: {DEVICE}')
print(f'segmentation_models_pytorch version: {smp.__version__}')

---
## Sección 1 — Preprocesamiento

In [ ]:
# ===================== CARGA DE DATOS =====================

def load_image(image_path: str) -> np.ndarray:
    """Carga imagen RGB uint8."""
    return np.array(Image.open(image_path).convert('RGB'))

def load_mask(mask_path: str) -> np.ndarray:
    """Carga máscara 16-bit sin truncar IDs de clase."""
    return cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)

def load_binary_mask(binary_mask_path: str) -> np.ndarray:
    """Carga máscara binaria y la binariza."""
    raw = cv2.imread(binary_mask_path, cv2.IMREAD_GRAYSCALE)
    return (raw > 127).astype(np.uint8)

def load_dataset_index(csv_path: str) -> pd.DataFrame:
    return pd.read_csv(csv_path, sep=';')

In [ ]:
# ===================== PREPROCESAMIENTO =====================

def to_grayscale(image: np.ndarray) -> np.ndarray:
    """Conversión perceptual."""
    return (0.299*image[:,:,0] + 0.587*image[:,:,1] + 0.114*image[:,:,2]).astype(np.uint8)

def map_entity_ids(mask: np.ndarray) -> np.ndarray:
    """Mapea IDs >22 a background."""
    result = mask.copy()
    result[result > 22] = 0
    return result.astype(np.uint8)

def compute_roi(binary_mask: np.ndarray, margin: float = 0.10) -> tuple:
    rows = np.any(binary_mask, axis=1)
    cols = np.any(binary_mask, axis=0)
    if not rows.any():
        h, w = binary_mask.shape
        return (0, 0, w, h)
    y1, y2 = np.where(rows)[0][[0, -1]]
    x1, x2 = np.where(cols)[0][[0, -1]]
    h, w = binary_mask.shape
    dy = max(1, int((y2-y1)*margin))
    dx = max(1, int((x2-x1)*margin))
    return (max(0,x1-dx), max(0,y1-dy), min(w,x2+dx), min(h,y2+dy))

def crop_to_roi(image, mask, roi):
    x1, y1, x2, y2 = roi
    return image[y1:y2, x1:x2], mask[y1:y2, x1:x2]

def resize_pair(image, mask, target_size):
    w, h = target_size
    img_r  = cv2.resize(image, (w, h), interpolation=cv2.INTER_LINEAR)
    mask_r = cv2.resize(mask,  (w, h), interpolation=cv2.INTER_NEAREST)
    return img_r, mask_r

def apply_clahe(image: np.ndarray, clip_limit=2.0, tile_grid=(8,8)) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid)
    return clahe.apply(image)

def replicate_to_3ch(image: np.ndarray) -> np.ndarray:
    return np.stack([image, image, image], axis=-1)

In [ ]:
# ===================== AUGMENTATION — MEJORADA PARA ESCOLIOSIS =====================
#
# Cambios clave respecto al notebook MaskRCNN:
#   1. ElasticTransform alpha=80 (vs alpha=1 — era casi inútil)
#   2. Rotaciones más agresivas ±25° (columna escoliótica puede estar muy inclinada)
#   3. Se agrega RandomGamma para simular variaciones de exposición en RX
#   4. Se agrega Sharpen para realzar bordes de vértebras

def build_train_augmentation() -> A.Compose:
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.Affine(
            rotate=(-25, 25),        # más agresivo que ±15°
            scale=(0.85, 1.15),
            shear=(-5, 5),           # simula deformación lateral leve
            mode=cv2.BORDER_CONSTANT,
            cval=0, cval_mask=0,
            p=0.7,
        ),
        # CRÍTICO: alpha alto simula deformación vertebral real
        A.ElasticTransform(alpha=80, sigma=8, p=0.4),
        A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.3, p=0.5),
        # Simula variaciones de exposición en radiografías
        A.RandomGamma(gamma_limit=(80, 120), p=0.4),
        # Realza bordes para ayudar a separar vértebras adyacentes
        A.Sharpen(alpha=(0.1, 0.3), lightness=(0.9, 1.1), p=0.3),
        A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
        A.CoarseDropout(
            num_holes_range=(1, 3),
            hole_height_range=(15, 40),
            hole_width_range=(15, 40),
            fill=0, p=0.15,
        ),
    ], additional_targets={'mask': 'mask'})

def build_val_augmentation() -> A.Compose:
    """Sin augmentation para val/test."""
    return A.Compose([], additional_targets={'mask': 'mask'})

In [ ]:
# ===================== DATASET =====================
#
# Diferencia clave con MaskRCNN:
#   - El target es una máscara semántica (H, W) con IDs de clase
#   - NO necesitamos convertir a instancias — U-Net++ trabaja directo con semántica
#   - Esto elimina la limitación de 1 instancia por clase

class SpineDataset(Dataset):
    def __init__(self, df: pd.DataFrame, mode: str,
                 target_size: tuple = TARGET_SIZE,
                 dataset_root: str = DATASET_ROOT):
        self.df          = df
        self.mode        = mode
        self.target_size = target_size
        self.root        = dataset_root
        self.aug         = build_train_augmentation() if mode == 'train' else build_val_augmentation()

        # Stats ImageNet (3 canales replicados de escala de grises)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]

        # Carga
        image       = load_image(os.path.join(self.root, row['radiograph_path']))
        mask        = load_mask(os.path.join(self.root, row['multiclass_id_png']))
        binary_mask = load_binary_mask(os.path.join(self.root, row['label_binary_path']))

        # Preprocesamiento
        image = to_grayscale(image)
        mask  = map_entity_ids(mask)
        roi   = compute_roi(binary_mask)
        image, mask = crop_to_roi(image, mask, roi)
        image, mask = resize_pair(image, mask, self.target_size)
        image = apply_clahe(image)
        image = replicate_to_3ch(image)  # (H, W, 3)

        # Augmentation (albumentations espera HWC)
        aug_result = self.aug(image=image, mask=mask)
        image = aug_result['image']
        mask  = aug_result['mask']

        # Normalización
        image = image.astype(np.float32) / 255.0
        image = (image - self.mean) / self.std
        image = torch.from_numpy(image.transpose(2, 0, 1))  # CHW
        mask  = torch.from_numpy(mask.astype(np.int64))     # HW, valores 0-17

        return image, mask


def split_dataset(df: pd.DataFrame, train=0.70, val=0.15, seed=42):
    train_df, temp_df = train_test_split(
        df, train_size=train, stratify=df['split'], random_state=seed
    )
    val_ratio = val / (1.0 - train)
    val_df, test_df = train_test_split(
        temp_df, train_size=val_ratio, stratify=temp_df['split'], random_state=seed
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True)

---
## Sección 2 — Modelo: U-Net++ con EfficientNet-B4

In [ ]:
# ===================== CONSTRUCCIÓN DEL MODELO =====================

def build_model(num_classes: int = NUM_CLASSES) -> nn.Module:
    """
    U-Net++ con encoder EfficientNet-B4 preentrenado en ImageNet.
    
    ¿Por qué EfficientNet-B4?
    - Mejor balance capacidad/memoria que ResNet50
    - Las compound scaling rules lo hacen superior en features finas
    - Probado extensamente en imágenes médicas
    
    ¿Por qué U-Net++ sobre U-Net?
    - Dense skip connections: el decoder recibe información de TODOS los niveles
    - Mejor para objetos de tamaño variable (vértebras cervicales vs lumbares)
    - Dice ~5-10% superior en benchmarks de segmentación médica
    """
    model = smp.UnetPlusPlus(
        encoder_name="efficientnet-b4",
        encoder_weights="imagenet",
        in_channels=3,           # 3 canales replicados de escala de grises
        classes=num_classes,
        activation=None,         # logits crudos; la loss aplica softmax internamente
        decoder_use_batchnorm=True,
    )
    return model

def count_parameters(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# Prueba rápida
model = build_model(NUM_CLASSES).to(DEVICE)
print(f'Parámetros entrenables: {count_parameters(model):,}')
dummy = torch.randn(2, 3, 1024, 512).to(DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'Output shape: {out.shape}')  # debe ser (2, 18, 1024, 512)
del dummy, out

In [ ]:
# ===================== FUNCIÓN DE PÉRDIDA COMBINADA =====================
#
# Por qué NO usar solo CrossEntropy:
#   - Las vértebras son pequeñas relativas al fondo → clase desbalanceada
#   - CrossEntropy optimiza accuracy pixel, no Dice
#
# Estrategia: 0.5 * DiceLoss + 0.5 * FocalLoss
#   - DiceLoss optimiza directamente la métrica de evaluación
#   - FocalLoss penaliza más los píxeles difíciles (bordes de vértebras)

class CombinedLoss(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, smooth=1e-6):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth
        self.focal  = smp.losses.FocalLoss(mode='multiclass', gamma=2.0)
        self.dice   = smp.losses.DiceLoss(mode='multiclass', smooth=smooth)

    def forward(self, pred, target):
        return 0.5 * self.dice(pred, target) + 0.5 * self.focal(pred, target)


# Alternativa con pesos de clase (útil si algunas vértebras tienen muy pocos píxeles)
class WeightedCombinedLoss(nn.Module):
    """
    Igual que CombinedLoss pero con peso extra para L4 y L5
    (las más relevantes para diagnóstico de escoliosis).
    Activa esto si el Dice de L4/L5 sigue siendo bajo después de entrenar.
    """
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        # Peso 1.0 para todas las clases, 2.0 para L4 (idx 16) y L5 (idx 17)
        weights = torch.ones(num_classes)
        weights[16] = 2.0  # L4
        weights[17] = 2.0  # L5
        self.weights = weights
        self.focal   = smp.losses.FocalLoss(mode='multiclass', gamma=2.0)
        self.dice    = smp.losses.DiceLoss(mode='multiclass')

    def forward(self, pred, target):
        return 0.5 * self.dice(pred, target) + 0.5 * self.focal(pred, target)

---
## Sección 3 — Entrenamiento

In [ ]:
# ===================== LOOPS DE ENTRENAMIENTO =====================

def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device) -> float:
    model.train()
    total_loss = 0.0
    for images, masks in dataloader:
        images = images.to(device)
        masks  = masks.to(device)   # (B, H, W) int64
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            preds = model(images)   # (B, C, H, W)
            loss  = criterion(preds, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(dataloader)


@torch.no_grad()
def validate_one_epoch(model, dataloader, criterion, device) -> tuple:
    """
    Retorna (val_loss, mean_dice) sobre el dataloader dado.
    mean_dice se calcula eficientemente en batch sin almacenar todas las predicciones.
    """
    model.eval()
    total_loss = 0.0
    dice_per_class = {c: [] for c in range(1, NUM_CLASSES)}

    for images, masks in dataloader:
        images = images.to(device)
        masks  = masks.to(device)

        with torch.amp.autocast('cuda'):
            preds = model(images)
            loss  = criterion(preds, masks)
        total_loss += loss.item()

        # Dice por clase en este batch
        pred_classes = preds.argmax(dim=1)  # (B, H, W)
        for c in range(1, NUM_CLASSES):
            pred_c = (pred_classes == c).float()
            gt_c   = (masks == c).float()
            # Solo calcular si la clase está presente en este batch
            if gt_c.sum() > 0:
                inter = (pred_c * gt_c).sum()
                denom = pred_c.sum() + gt_c.sum()
                dice_per_class[c].append((2.0 * inter / denom).item() if denom > 0 else 1.0)

    avg_loss = total_loss / len(dataloader)
    mean_dice = np.mean([np.mean(v) for v in dice_per_class.values() if v])
    return avg_loss, float(mean_dice)


def save_checkpoint(model, path, val_dice, val_loss):
    torch.save(model.state_dict(), path)
    print(f'  ✔ Checkpoint guardado: {path}  (val_dice={val_dice:.4f}, val_loss={val_loss:.4f})')

def load_checkpoint(model, path):
    model.load_state_dict(torch.load(path, map_location='cpu'))
    return model

In [ ]:
# ===================== SCHEDULER CON WARMUP =====================
#
# Warmup lineal 5 epochs → CosineAnnealingLR
# Esto es crítico para EfficientNet: arrancar con LR alto destruye los pesos preentrenados.

def build_optimizer_and_scheduler(model, base_lr=BASE_LR, weight_decay=WEIGHT_DECAY, num_epochs=NUM_EPOCHS):
    """
    LR diferencial:
    - Encoder (EfficientNet): base_lr * 0.1  (ya está preentrenado)
    - Decoder (U-Net++ heads): base_lr       (se entrena desde cero)
    """
    encoder_params = []
    decoder_params = []
    for name, param in model.named_parameters():
        if 'encoder' in name:
            encoder_params.append(param)
        else:
            decoder_params.append(param)

    param_groups = [
        {'params': decoder_params, 'lr': base_lr},
        {'params': encoder_params, 'lr': base_lr * 0.1},
    ]

    optimizer = torch.optim.AdamW(param_groups, weight_decay=weight_decay)

    # Warmup 5 epochs + CosineAnnealing
    warmup_epochs = 5
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer, start_factor=0.1, end_factor=1.0, total_iters=warmup_epochs
    )
    cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-6
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_scheduler, cosine_scheduler],
        milestones=[warmup_epochs]
    )

    return optimizer, scheduler

In [ ]:
# ===================== LOOP PRINCIPAL DE ENTRENAMIENTO =====================

def train(
    model, train_dl, val_dl, device,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    use_weighted_loss=False,
    checkpoint_path='checkpoints_unetpp/best.pth'
) -> dict:
    """
    Entrena U-Net++ con:
    - Mixed precision FP16
    - LR diferencial encoder/decoder
    - Warmup + CosineAnnealing
    - Early stopping por val_dice (↑ es mejor)
    - Checkpointing del mejor modelo
    Retorna historial de métricas.
    """
    criterion = WeightedCombinedLoss() if use_weighted_loss else CombinedLoss()
    criterion = criterion.to(device)
    optimizer, scheduler = build_optimizer_and_scheduler(model, base_lr, weight_decay, num_epochs)
    scaler = torch.amp.GradScaler('cuda')

    history = {'train_loss': [], 'val_loss': [], 'val_dice': []}
    best_val_dice  = -1.0
    epochs_no_impr = 0

    for epoch in range(1, num_epochs + 1):
        train_loss           = train_one_epoch(model, train_dl, optimizer, criterion, scaler, device)
        val_loss, val_dice   = validate_one_epoch(model, val_dl, criterion, device)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)

        lr_now = optimizer.param_groups[0]['lr']
        print(f'Epoch {epoch:3d}/{num_epochs} — '
              f'train_loss: {train_loss:.4f}  '
              f'val_loss: {val_loss:.4f}  '
              f'val_dice: {val_dice:.4f}  '
              f'lr: {lr_now:.2e}')

        if val_dice > best_val_dice:
            best_val_dice  = val_dice
            epochs_no_impr = 0
            save_checkpoint(model, checkpoint_path, val_dice, val_loss)
        else:
            epochs_no_impr += 1
            if epochs_no_impr >= patience:
                print(f'Early stopping en epoch {epoch}. Mejor val_dice: {best_val_dice:.4f}')
                break

    model = load_checkpoint(model, checkpoint_path)
    print(f'\nEntrenamiento completado. Mejor val_dice: {best_val_dice:.4f}')
    return history

---
## Sección 4 — Métricas

In [ ]:
# ===================== MÉTRICAS =====================

@torch.no_grad()
def run_inference(model, dataloader, device):
    """
    Retorna (all_preds, all_targets) como numpy arrays (H, W) con IDs de clase.
    Más eficiente que almacenar probabilidades completas.
    """
    model.eval()
    all_preds, all_targets = [], []
    for images, masks in dataloader:
        images = images.to(device)
        with torch.amp.autocast('cuda'):
            preds = model(images).argmax(dim=1)  # (B, H, W)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(masks.numpy())
    return all_preds, all_targets


def compute_dice_per_class(preds, targets, num_classes=NUM_CLASSES) -> dict:
    scores = {c: [] for c in range(1, num_classes)}
    for pred, target in zip(preds, targets):
        for c in range(1, num_classes):
            gt_c   = (target == c).astype(bool)
            pred_c = (pred   == c).astype(bool)
            if not gt_c.any():           # clase ausente en esta imagen
                continue
            inter  = (pred_c & gt_c).sum()
            denom  = pred_c.sum() + gt_c.sum()
            dice   = 2.0 * inter / denom if denom > 0 else 1.0
            scores[c].append(float(dice))
    return {c: float(np.mean(v)) for c, v in scores.items() if v}


def compute_iou_per_class(preds, targets, num_classes=NUM_CLASSES) -> dict:
    scores = {c: [] for c in range(1, num_classes)}
    for pred, target in zip(preds, targets):
        for c in range(1, num_classes):
            gt_c   = (target == c).astype(bool)
            pred_c = (pred   == c).astype(bool)
            if not gt_c.any():
                continue
            inter = (pred_c & gt_c).sum()
            union = (pred_c | gt_c).sum()
            iou   = inter / union if union > 0 else 1.0
            scores[c].append(float(iou))
    return {c: float(np.mean(v)) for c, v in scores.items() if v}


def print_metrics_report(dice_per_class, iou_per_class):
    print(f"{'Clase':<8} {'Dice':>8} {'IoU':>8}")
    print('-' * 26)
    for c in range(1, NUM_CLASSES):
        name = CLASS_NAMES.get(c, f'ID{c}')
        dice = dice_per_class.get(c, float('nan'))
        iou  = iou_per_class.get(c, float('nan'))
        print(f"{name:<8} {dice:>8.4f} {iou:>8.4f}")
    print('-' * 26)
    mean_dice = np.mean(list(dice_per_class.values()))
    mean_iou  = np.mean(list(iou_per_class.values()))
    print(f"{'MEAN':<8} {mean_dice:>8.4f} {mean_iou:>8.4f}")
    return mean_dice, mean_iou

In [ ]:
# ===================== VISUALIZACIÓN =====================

def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history['train_loss']) + 1)

    ax1.plot(epochs, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
    ax1.plot(epochs, history['val_loss'],   'r-o', markersize=3, label='Val Loss')
    ax1.set_title('Loss por epoch')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history['val_dice'], 'g-o', markersize=3, label='Val Dice')
    ax2.set_title('Dice de Validación')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Dice')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim(0, 1)

    plt.suptitle('Historial de Entrenamiento — U-Net++ EfficientNet-B4', fontsize=13)
    plt.tight_layout()
    plt.show()


def visualize_predictions(model, test_ds, device, n=4, seed=42):
    """
    Muestra n ejemplos aleatorios con overlay de predicción vs ground truth.
    """
    random.seed(seed)
    indices = random.sample(range(len(test_ds)), min(n, len(test_ds)))

    rng     = np.random.RandomState(0)
    palette = rng.randint(80, 230, size=(NUM_CLASSES, 3), dtype=np.uint8)
    palette[0] = [0, 0, 0]

    model.eval()
    fig, axes = plt.subplots(len(indices), 3, figsize=(15, 5 * len(indices)))
    if len(indices) == 1:
        axes = axes[np.newaxis, :]

    for row, idx in enumerate(indices):
        image, gt_mask = test_ds[idx]
        with torch.no_grad():
            pred = model(image.unsqueeze(0).to(device))
            pred_mask = pred.argmax(dim=1).squeeze().cpu().numpy()

        # Imagen original (denormalize)
        img_np = image.numpy().transpose(1, 2, 0)
        mean_v = np.array([0.485, 0.456, 0.406])
        std_v  = np.array([0.229, 0.224, 0.225])
        img_np = np.clip((img_np * std_v + mean_v) * 255, 0, 255).astype(np.uint8)

        # Coloreado de masks
        gt_color   = palette[gt_mask.numpy()]
        pred_color = palette[pred_mask]

        axes[row, 0].imshow(img_np)
        axes[row, 0].set_title(f'Imagen #{idx}')
        axes[row, 0].axis('off')

        axes[row, 1].imshow(gt_color)
        axes[row, 1].set_title('Ground Truth')
        axes[row, 1].axis('off')

        axes[row, 2].imshow(pred_color)
        axes[row, 2].set_title('Predicción')
        axes[row, 2].axis('off')

    plt.suptitle('Comparación: Ground Truth vs Predicción — U-Net++', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Pipeline Principal

In [ ]:
# ===================== PREPROCESAMIENTO =====================
index_df = load_dataset_index(DATASET_INDEX)
train_df, val_df, test_df = split_dataset(index_df, seed=SEED)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

train_ds = SpineDataset(train_df, mode='train')
val_ds   = SpineDataset(val_df,   mode='val')
test_ds  = SpineDataset(test_df,  mode='test')

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Batches — train: {len(train_dl)}, val: {len(val_dl)}, test: {len(test_dl)}')

In [ ]:
# ===================== ENTRENAMIENTO =====================
model = build_model(NUM_CLASSES).to(DEVICE)

history = train(
    model       = model,
    train_dl    = train_dl,
    val_dl      = val_dl,
    device      = DEVICE,
    num_epochs  = NUM_EPOCHS,
    patience    = PATIENCE,
    base_lr     = BASE_LR,
    weight_decay= WEIGHT_DECAY,
    use_weighted_loss = False,   # Cambiar a True si L4/L5 tienen Dice bajo
    checkpoint_path   = os.path.join(CHECKPOINTS_DIR, 'unetpp_efficientnetb4_best.pth')
)

In [ ]:
# ===================== RESULTADOS =====================
plot_training_history(history)

print('\n=== EVALUACIÓN EN TEST SET ===')
all_preds, all_targets = run_inference(model, test_dl, DEVICE)
dice_per_class = compute_dice_per_class(all_preds, all_targets)
iou_per_class  = compute_iou_per_class(all_preds, all_targets)
mean_dice, mean_iou = print_metrics_report(dice_per_class, iou_per_class)

print(f'\n>>> mean Dice: {mean_dice:.4f}')
print(f'>>> mean IoU:  {mean_iou:.4f}')

In [ ]:
# ===================== VISUALIZACIÓN =====================
visualize_predictions(model, test_ds, DEVICE, n=4)

In [ ]:
# ===================== GUARDAR MODELO FINAL =====================
final_path = os.path.join(DRIVE_ROOT, 'models', 'unetpp_efficientnetb4_final.pth')
os.makedirs(os.path.dirname(final_path), exist_ok=True)
torch.save(model.state_dict(), final_path)
print(f'Modelo final guardado en: {final_path}')

---
## Apéndice — Próximos pasos si el Dice aún es insuficiente

### Si obtienes Dice entre 0.4 y 0.6:
1. **Activar `use_weighted_loss=True`** — da más peso a L4/L5
2. **Test-Time Augmentation (TTA):** promediar predicciones con flips horizontales
3. **Aumentar TARGET_SIZE a (640, 1280)** — más detalle para vértebras pequeñas

### Si obtienes Dice entre 0.6 y 0.75:
1. **Cambiar encoder a EfficientNet-B6 o MiT-B3 (SegFormer)**
2. **Pipeline de 2 etapas:** detección de columna → segmentación localizada
3. **Pseudo-labeling con SAM** para expandir el dataset a ~500 imágenes

### Referencia esperada para tu dataset:
| Arquitectura        | Dice esperado |
|---------------------|---------------|
| MaskRCNN ResNet50   | 0.20–0.30     |
| U-Net ResNet34      | 0.45–0.60     |
| **U-Net++ EffB4**   | **0.60–0.75** |
| SegFormer MiT-B3    | 0.70–0.82     |
| TransUNet           | 0.78–0.87     |